# A Simulation of the BB84 Quantum Key Distribution (QKD) Protocol


## 1. Setup & Utilities

- Import necessary packages

In [ ]:
import qiskit
import math
import random
import numpy as np
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister, transpile
from qiskit.primitives import StatevectorSampler
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler

from qiskit.primitives import BackendSamplerV2
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, pauli_error

- Save credentials locally. This is one-time setup, does not need to re-run every times we build the notebook.

In [ ]:
# from qiskit_ibm_runtime import QiskitRuntimeService
# Stores credentials locally
# one-time setup
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<token>",
#     overwrite=True,
#     set_as_default=True
# )


- Helpers for experiments

In [ ]:
def make_random_number(seed=None):
    """
    This helper create a random number generator

    Params:
        seed: Random seed for reproducibility. If None, results vary each run.

    Returns:
         A random number generator
    """
    return np.random.default_rng(seed)

def sample_bits(bit_num, rng):
    """Sample an array of random 0|1 bits."""
    return rng.integers(0, 2, size=bit_num)

def encode_bb84_states(qc, bits, bases):
    """
    Encode BB84 states onto the qubits in-place.

    Convention:
    bit=0, basis=0 -> |0>
    bit=1, basis=0 -> |1>
    bit=0, basis=1 -> |+>
    bit=1, basis=1 -> |->
    """
    bit_num = len(bits)

    for n in range(bit_num):
        if bits[n] == 0:
            if bases[n] == 1:
                qc.h(n)
        if bits[n] == 1:
            if bases[n] == 0:
                qc.x(n)
            if bases[n] == 1:
                qc.x(n)
                qc.h(n)

def measure_bb84(qc, bases):
    """
    Measure qubits in BB84 bases in-place.

    basis=0 -> Z measurement
    basis=1 -> X measurement via H then Z measurement
    """
    bit_num = len(bases)

    for n in range(bit_num):
        if bases[n] == 1:
            qc.h(n)
        qc.measure(n, n)

def run_statevector_sampler(qc, shots=1):
    """
    Run a circuit locally with StatevectorSampler.
    """
    sampler = StatevectorSampler()
    return sampler.run([qc], shots=shots).result()


def extract_counts(result):
    """
    Extract both string and integer counts from result.
    """
    counts = result[0].data.c.get_counts()
    countsint = result[0].data.c.get_int_counts()
    return counts, countsint


def counts_to_bbits(counts, bit_num):
    """
    Convert the first measured bitstring in counts to Bob's bit array.

    Assumes shots=1 or that the first key is the one you want to inspect.
    Reverses bit order to match qubit indexing.
    """
    keys = counts.keys()
    key = list(keys)[0]
    bmeas = list(key)

    bmeas_ints = []
    for n in range(bit_num):
        bmeas_ints.append(int(bmeas[n]))

    bbits = bmeas_ints[::-1]
    return bbits


def sift_key(sender_bits, sender_bases, receiver_bases, receiver_bits):
    """
    Keep only positions where sender and receiver bases match.
    """
    sender_sifted = []
    receiver_sifted = []

    for n in range(len(sender_bits)):
        if sender_bases[n] == receiver_bases[n]:
            sender_sifted.append(int(sender_bits[n]))
            receiver_sifted.append(int(receiver_bits[n]))

    return sender_sifted, receiver_sifted


def key_statistics(sender_sifted, receiver_sifted):
    """
    Compute match count, fidelity, and loss for sifted keys.
    """
    if len(sender_sifted) == 0:
        return {
            "match_count": 0,
            "sifted_length": 0,
            "fidelity": None,
            "loss": None,
        }

    match_count = 0
    for n in range(len(sender_sifted)):
        if sender_sifted[n] == receiver_sifted[n]:
            match_count += 1

    fidelity = match_count / len(sender_sifted)
    loss = 1 - fidelity

    return {
        "match_count": match_count,
        "sifted_length": len(sender_sifted),
        "fidelity": fidelity,
        "loss": loss,
    }


def print_protocol_summary(abits, abase, bbase, bbits, agoodbits, bgoodbits, stats):
    """
    Print a compact summary for one BB84-like run.
    """
    print("Alice's bits are ", list(abits))
    print("Alice's bases are ", list(abase))
    print("Bob's bases are   ", list(bbase))
    print("Bob's measured bits:", bbits)
    print("Alice sifted key:", agoodbits)
    print("Bob sifted key:  ", bgoodbits)
    print("fidelity = ", stats["fidelity"])
    print("loss = ", stats["loss"])
    


## 2. Experiment 1: Ideal BB84 QKD on IBM hardware - No Noise, No Eve
This experiment simulate how Alice sends qubits and Bob measures them using random bases to generate a shared secret key.
- Alice encodes random bits into qubits using random bases (Z or X)
- Bob measures using random bases
- Later, they compare bases and keep only matching ones

In [ ]:
# Setup
bit_num = 20
rng = make_random_number(123)
abits = sample_bits(bit_num, rng)
abase = sample_bits(bit_num, rng)
bbase = sample_bits(bit_num, rng)
# 20 qubits, 20 classical bits for measurement results
qc = QuantumCircuit(bit_num, bit_num)

# Encoding: Alice prepares qubits
encode_bb84_states(qc, abits, abase)
qc.barrier()

# Bob measures in his chosen bases
measure_bb84(qc, bbase)

display(qc.draw("mpl"))

# Connect to IBM Quantum hardware
service = QiskitRuntimeService()

backend = service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=127
)

print("Backend:", backend.name)


# Transpile for the backend
pm = generate_preset_pass_manager(
    target=backend.target,
    optimization_level=1
)
qc_isa = pm.run(qc)

# Execute on hardware
sampler = Sampler(mode=backend)
job = sampler.run([qc_isa], shots=1)

print("Job ID:", job.job_id())

result = job.result()


# Post-process
counts, countsint = extract_counts(result)
bbits = counts_to_bbits(counts, bit_num)

agoodbits, bgoodbits = sift_key(abits, abase, bbase, bbits)
stats = key_statistics(agoodbits, bgoodbits)

print_protocol_summary(abits, abase, bbase, bbits, agoodbits, bgoodbits, stats)

## 3. Experiment 2: BB84 QKD with Channel Noise
Make the noise represent the transmission channel by:


## 4. Experiment 3: BB84 QKD with Eavesdropper (Intercept-Resend)
First attack model:
- Eve random basis and measure qubit
- Eve resend it to Bob
- QBER raises ~25% theoretical

## 5. Experiment 4: BB84 QKD with Noise + Eve Combined

## 6. Experiment 5: Error correction
- Classical post-processing
- parity check
- Show QBER decreases after correction

## 7. Experiment 6: Privacy Amplification
- Hashing SHA
- Compress key
- Final key could be shorter but secure b/c Eve information decreases 

In [ ]:
References:
https://quantum.cloud.ibm.com/learning/en/modules/computer-science/quantum-key-distribution